# Transformer S3
Compact PyTorch version with explicit attention, training, inference, and quantization/sparsity hooks.

In [ ]:
import torch,random,math,re,json,subprocess
from pathlib import Path
from collections import Counter
import torch.nn as nn
import torch.nn.functional as F
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg={
    "d_model":64,
    "d_ff":128,
    "num_heads":4,
    "num_enc":2,
    "num_dec":2,
    "max_vocab":50000,
    "max_words":32,
    "batch_size":4,
    "lr":0.001,
    "use_wq":False,
    "use_aq":False,
    "w_bits":8,
    "a_bits":8,
    "use_ws":False,
    "ws_ratio":0.0,
    "attn_top_k":None,
    "norm":"post",
    "epochs":100
}
print(device)

cuda


## Data

#### Load BabiStories Dataset

In [50]:
def read_texts_from_file(p):
    texts=[]
    if p.suffix.lower()==".txt":
        x=p.read_text(encoding="utf-8",errors="ignore")
        parts=re.split(r"\n\s*\n|\n",x)
        for t in parts:
            t=" ".join(t.split())
            if len(t.split())>20:texts.append(t)
    elif p.suffix.lower()==".jsonl":
        for line in p.open(encoding="utf-8",errors="ignore"):
            if line.strip():
                d=json.loads(line)
                t=d.get("text") or d.get("story") or d.get("content") or ""
                t=" ".join(t.split())
                if len(t.split())>20:texts.append(t)
    elif p.suffix.lower()==".json":
        d=json.loads(p.read_text(encoding="utf-8",errors="ignore"))
        if isinstance(d,list):
            for e in d:
                t=e.get("text") or e.get("story") or e.get("content") or ""
                t=" ".join(t.split())
                if len(t.split())>20:texts.append(t)
    return texts

def load_babistories_from_extracted(folder):
    folder=Path(folder)
    texts=[]
    for p in folder.rglob("*"):
        if p.suffix.lower() in [".txt",".jsonl",".json"]:
            texts+=read_texts_from_file(p)
    return texts

texts=load_babistories_from_extracted("BabiStories/data/extracted")
print("texts:",len(texts))
print(texts[0][:500])

texts: 2221391
"Every year, the town of Meadowville held a big fair, and Dex loved it! This year, there was a new ride that everyone was talking about: the \"Hanging Swings.\" Dex was small, but brave, and begged his mom to let him try it. \"Okay Dex, but hold on tight!\" his mom said with a smile. Up in the air, Dex hung from the swing, laughing with joy. He saw something cute on the ground - a little kitten was looking at him! He pointed his foot down to say \"hi\" to the kitten. \"Meow!\" said the kitten so


##### Convering stories into encoder-decoder pairs:

In [51]:
def make_babistory_pairs(texts,src_words=32,tgt_words=32,stride=16,max_pairs=5000):
    pairs=[]
    for text in texts:
        tokens=text.split()
        if len(tokens)<src_words+tgt_words:continue
        for i in range(0,len(tokens)-src_words-tgt_words+1,stride):
            s=" ".join(tokens[i:i+src_words])
            t=" ".join(tokens[i+src_words:i+src_words+tgt_words])
            pairs.append((s,t))
            if len(pairs)>=max_pairs:return pairs
    return pairs

# cfg["max_words"]=32
pairs=make_babistory_pairs(texts,src_words=cfg["max_words"],tgt_words=cfg["max_words"],stride=16,max_pairs=5000)
random.shuffle(pairs)
n=int(0.9*len(pairs))
train_data=pairs[:n]
valid_data=pairs[n:]
print("pairs:",len(pairs))
print("train:",len(train_data))
print("valid:",len(valid_data))
print("source:",train_data[0][0])
print("target:",train_data[0][1])

KeyError: 'max_words'

In [ ]:
# def prepare_pairs(raw,max_words=20,limit=3000):
#     pairs=[]
#     for e in raw:
#         s=e["source"].strip();t=e["target"].strip()
#         if s and t and len(s.split())<=max_words and len(t.split())<=max_words:pairs.append((s,t))
#         if len(pairs)>=limit:break
#     return pairs
def build_context(pairs,cfg):
    sp=["<PAD>","<SOS>","<EOS>","<UNK>"];c=Counter()
    for s,t in pairs:c.update(s.split());c.update(t.split())
    vocab=sp+[w for w,_ in c.most_common(cfg["max_vocab"]-len(sp))]
    stoi={w:i for i,w in enumerate(vocab)};itos={i:w for w,i in stoi.items()}
    return {"stoi":stoi,"itos":itos,"vocab":vocab,"pad":stoi["<PAD>"],"sos":stoi["<SOS>"],"eos":stoi["<EOS>"],"unk":stoi["<UNK>"],"src_len":cfg["max_words"],"tgt_len":cfg["max_words"]+1}
def encode(text,ctx,max_len,add_sos=False,add_eos=False):
    ids=[ctx["stoi"].get(w,ctx["unk"]) for w in text.split()]
    if add_sos:ids=[ctx["sos"]]+ids
    if add_eos:ids=ids+[ctx["eos"]]
    ids=ids[:max_len];ids=ids+[ctx["pad"]]*(max_len-len(ids))
    return ids
# def pad_batch(sequences,pad_id):
#     max_len=max(len(seq) for seq in sequences)
#     batch=[]
#     for seq in sequences:
#         padded_seq=seq+[pad_id]*(max_len-len(seq))
#         batch.append(padded_seq)
#     return torch.tensor(batch,dtype=torch.long)
def make_batch(data,ctx,batch_size,device):
    if len(data)==0:raise ValueError("data is empty")
    if len(data)<batch_size:b=random.choices(data,k=batch_size)
    else:b=random.sample(data,batch_size)
    src=torch.tensor([encode(s,ctx,ctx["src_len"]) for s,t in b],dtype=torch.long,device=device)
    dec=torch.tensor([encode(t,ctx,ctx["tgt_len"],add_sos=True) for s,t in b],dtype=torch.long,device=device)
    y=torch.tensor([encode(t,ctx,ctx["tgt_len"],add_eos=True) for s,t in b],dtype=torch.long,device=device)
    return src,dec,y,src.eq(ctx["pad"]),dec.eq(ctx["pad"])

def acc_ignore_pad(logits,y,pad_id):
    pred=logits.argmax(-1)
    mask=y.ne(pad_id)
    correct=pred.eq(y).logical_and(mask).sum().item()
    total=mask.sum().item()
    return correct/total

## Quantization and sparsity hooks

In [ ]:
def qste(x,bits,use):
    if not use:return x
    qmax=2**(bits-1)-1;s=x.abs().max().clamp(min=1e-8)/qmax
    y=(x/s).round().clamp(-qmax,qmax)*s
    return x+(y-x).detach()
def sparsify(w,ratio,use):
    if not use or ratio<=0:return w
    th=torch.quantile(w.abs().flatten(),ratio)
    return w*(w.abs()>=th)
def qw(w,cfg):return qste(sparsify(w,cfg["ws_ratio"],cfg["use_ws"]),cfg["w_bits"],cfg["use_wq"])
def qa(x,cfg):return qste(x,cfg["a_bits"],cfg["use_aq"])
def lin(x,l,cfg):return F.linear(qa(x,cfg),qw(l.weight,cfg),l.bias)

## Model

In [ ]:
# ------------------------------------------
# Multihead-Attention Class
# ------------------------------------------
class MHA(nn.Module):

    def __init__(self,d_model,num_heads,cfg):
        super().__init__();self.h=num_heads;self.dh=d_model//num_heads;self.cfg=cfg
        self.q=nn.Linear(d_model,d_model);self.k=nn.Linear(d_model,d_model);self.v=nn.Linear(d_model,d_model);self.o=nn.Linear(d_model,d_model)

    def forward(self,q_in,kv_in,causal=False,key_pad=None):
        B,Tq,D=q_in.shape;Tk=kv_in.shape[1]
        Q=lin(q_in,self.q,self.cfg).view(B,Tq,self.h,self.dh).transpose(1,2)
        K=lin(kv_in,self.k,self.cfg).view(B,Tk,self.h,self.dh).transpose(1,2)
        V=lin(kv_in,self.v,self.cfg).view(B,Tk,self.h,self.dh).transpose(1,2)
        S=Q@K.transpose(-2,-1)/math.sqrt(self.dh)
        if key_pad is not None:S=S.masked_fill(key_pad[:,None,None,:],-1e9)
        if causal:S=S.masked_fill(torch.triu(torch.ones(Tq,Tk,device=q_in.device,dtype=torch.bool),1)[None,None,:,:],-1e9)
        if self.cfg["attn_top_k"] is not None:
            k=min(self.cfg["attn_top_k"],Tk);th=S.topk(k,dim=-1).values[...,-1,None];S=S.masked_fill(S<th,-1e9)
        A=F.softmax(S,dim=-1);O=(A@V).transpose(1,2).contiguous().view(B,Tq,D)
        return lin(O,self.o,self.cfg),A

# ------------------------------------------
# Feed-Forward Network Class
# ------------------------------------------    
class FFN(nn.Module):
    def __init__(self,d_model,d_ff,cfg):
        super().__init__();self.l1=nn.Linear(d_model,d_ff);self.l2=nn.Linear(d_ff,d_model);self.cfg=cfg
    def forward(self,x):
        return lin(F.relu(lin(x,self.l1,self.cfg)),self.l2,self.cfg)
    
# ------------------------------------------
# Encoder Block Class
# ------------------------------------------      
class EncBlock(nn.Module):
    def __init__(self,d_model,d_ff,num_heads,cfg):
        super().__init__();self.a=MHA(d_model,num_heads,cfg);self.f=FFN(d_model,d_ff,cfg);self.n1=nn.LayerNorm(d_model);self.n2=nn.LayerNorm(d_model);self.cfg=cfg
    def forward(self,x,src_pad):
        if self.cfg["norm"]=="post":a,A=self.a(x,x,key_pad=src_pad);x=self.n1(x+a);x=self.n2(x+self.f(x))
        else:n=self.n1(x);a,A=self.a(n,n,key_pad=src_pad);x=x+a;x=x+self.f(self.n2(x))
        return x,A
    
# ------------------------------------------
# Decoder Block Class
# ------------------------------------------   
class DecBlock(nn.Module):
    def __init__(self,d_model,d_ff,num_heads,cfg):
        super().__init__();self.sa=MHA(d_model,num_heads,cfg);self.ca=MHA(d_model,num_heads,cfg);self.f=FFN(d_model,d_ff,cfg);self.n1=nn.LayerNorm(d_model);self.n2=nn.LayerNorm(d_model);self.n3=nn.LayerNorm(d_model);self.cfg=cfg
    def forward(self,x,enc,src_pad,tgt_pad):
        if self.cfg["norm"]=="post":s,SA=self.sa(x,x,causal=True,key_pad=tgt_pad);x=self.n1(x+s);c,CA=self.ca(x,enc,key_pad=src_pad);x=self.n2(x+c);x=self.n3(x+self.f(x))
        else:n=self.n1(x);s,SA=self.sa(n,n,causal=True,key_pad=tgt_pad);x=x+s;n=self.n2(x);c,CA=self.ca(n,enc,key_pad=src_pad);x=x+c;x=x+self.f(self.n3(x))
        return x,SA,CA

# ------------------------------------------
# Sequence to Sequence Class
# ------------------------------------------   
class Seq2Seq(nn.Module):
    def __init__(self,vocab_size,cfg,ctx):
        super().__init__();d=cfg["d_model"];self.cfg=cfg;self.ctx=ctx;self.tok=nn.Embedding(vocab_size,d,padding_idx=ctx["pad"]);self.pos=nn.Embedding(max(ctx["src_len"],ctx["tgt_len"]),d)
        self.enc=nn.ModuleList([EncBlock(d,cfg["d_ff"],cfg["num_heads"],cfg) for _ in range(cfg["num_enc"])])
        self.dec=nn.ModuleList([DecBlock(d,cfg["d_ff"],cfg["num_heads"],cfg) for _ in range(cfg["num_dec"])]);self.out=nn.Linear(d,vocab_size)
    def forward(self,src,dec,src_pad,tgt_pad):
        ps=torch.arange(src.size(1),device=src.device)[None,:];pt=torch.arange(dec.size(1),device=dec.device)[None,:]
        x=self.tok(src)+self.pos(ps);y=self.tok(dec)+self.pos(pt);EA=[];SA=[];CA=[]
        for b in self.enc:x,a=b(x,src_pad);EA.append(a)
        for b in self.dec:y,s,c=b(y,x,src_pad,tgt_pad);SA.append(s);CA.append(c)
        return self.out(y),EA,SA,CA

## Load dataset

In [ ]:
pairs=make_babistory_pairs(texts,src_words=cfg["max_words"],tgt_words=cfg["max_words"],stride=16,max_pairs=5000)
random.seed(42);random.shuffle(pairs)
n_train=int(0.8*len(pairs))
n_valid=int(0.1*len(pairs))
train_data=pairs[:n_train]
valid_data=pairs[n_train:n_train+n_valid]
test_data=pairs[n_train+n_valid:]
ctx=build_context(train_data,cfg)
print("pairs:",len(pairs))
print("train:",len(train_data))
print("valid:",len(valid_data))
print("test:",len(test_data))
print("vocab:",len(ctx["vocab"]))
print("source:",train_data[0][0])
print("target:",train_data[0][1])

pairs: 5000
train: 4000
valid: 500
test: 500
vocab: 11164
source: to him. The brush was shiny and bristly, and Gerry used it to keep his mane neat and tidy. One day, while Gerry was gazing at his brush, he noticed something unusual.
target: The bristles of the brush started to move on their own, forming words. \"Gerry, I am a magic brush! I can make your mane even more beautiful!\" Gerry was surprised but excited!


## Train/evaluate

In [ ]:
model=Seq2Seq(len(ctx["vocab"]),cfg,ctx).to(device)
opt=torch.optim.AdamW(model.parameters(),lr=cfg["lr"],weight_decay=0.01)

def acc_ignore_pad(logits,y,pad_id):
    pred=logits.argmax(-1)
    m=y.ne(pad_id)
    total=m.sum().item()
    if total==0:return 0.0
    correct=(pred.eq(y)&m).sum().item()
    return correct/total

def step(data,train=True):
    model.train(train)
    src,dec,y,src_pad,tgt_pad=make_batch(data,ctx,cfg["batch_size"],device)
    with torch.set_grad_enabled(train):
        logits,EA,SA,CA=model(src,dec,src_pad,tgt_pad)
        loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1),ignore_index=ctx["pad"])
        if train:
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()
    acc=acc_ignore_pad(logits,y,ctx["pad"])
    return loss.item(),acc

def evaluate(data,n=20):
    loss=0;acc=0
    for _ in range(n):
        l,a=step(data,False)
        loss+=l;acc+=a
    return loss/n,acc/n

In [ ]:
best_state=None
history={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}
def train_epochs(epochs=200,train_steps=100,valid_steps=50,patience=15):
    history={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}
    best=float("inf");best_state=None;bad=0
    for epoch in range(1,epochs+1):
        tl=ta=0
        for _ in range(train_steps):
            l,a=step(train_data,True)
            tl+=l;ta+=a
        vl,va=evaluate(valid_data,valid_steps)
        tl/=train_steps;ta/=train_steps
        history["train_loss"].append(tl);history["train_acc"].append(ta)
        history["val_loss"].append(vl);history["val_acc"].append(va)
        if vl<best:
            best=vl;bad=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            bad+=1
        print(epoch,"train_loss",tl,"train_acc",ta,"val_loss",vl,"val_acc",va)
        if bad>=patience:
            print("early stop at epoch",epoch,"best_val_loss",best)
            break
    return best_state,history

    # print(cfg["epochs"])
best_state,history=train_epochs(epochs=200,train_steps=100,valid_steps=50,patience=15)
model.load_state_dict(best_state)

1 train_loss 7.684883799552917 train_acc 0.04621212121212123 val_loss 7.167858400344849 val_acc 0.07348484848484851
2 train_loss 6.889091305732727 train_acc 0.08045454545454545 val_loss 6.884405269622802 val_acc 0.08106060606060607
3 train_loss 6.797439932823181 train_acc 0.08643939393939398 val_loss 6.876452989578247 val_acc 0.09484848484848482
4 train_loss 6.641488375663758 train_acc 0.09856060606060604 val_loss 6.759650688171387 val_acc 0.10060606060606063
5 train_loss 6.544138331413269 train_acc 0.10818181818181813 val_loss 6.671301679611206 val_acc 0.10863636363636363
6 train_loss 6.3781973361969 train_acc 0.1088636363636364 val_loss 6.59502984046936 val_acc 0.11787878787878789
7 train_loss 6.280418972969056 train_acc 0.12090909090909094 val_loss 6.433768558502197 val_acc 0.11803030303030304
8 train_loss 6.216453628540039 train_acc 0.12696969696969695 val_loss 6.462471389770508 val_acc 0.12121212121212119
9 train_loss 6.112076106071473 train_acc 0.13439393939393937 val_loss 6.3632

<All keys matched successfully>

## Inference

In [ ]:
def decode_ids(ids,ctx):
    bad={ctx["pad"],ctx["sos"],ctx["eos"]}
    return " ".join([ctx["itos"][int(i)] for i in ids if int(i) not in bad])

def generate(model,src_text,ctx,max_new=None):
    model.eval();max_new=max_new or ctx["tgt_len"]
    src=torch.tensor([encode(src_text,ctx,ctx["src_len"])],device=device);src_pad=src.eq(ctx["pad"]);ids=[ctx["sos"]]
    for _ in range(max_new):
        dec=ids+[ctx["pad"]]*(ctx["tgt_len"]-len(ids));dec=torch.tensor([dec[:ctx["tgt_len"]]],device=device);tgt_pad=dec.eq(ctx["pad"])
        with torch.no_grad():logits,EA,SA,CA=model(src,dec,src_pad,tgt_pad)
        nxt=int(logits[0,len(ids)-1].argmax())
        if nxt==ctx["eos"]:break
        ids.append(nxt)
        if len(ids)>=ctx["tgt_len"]:break
    return decode_ids(ids[1:],ctx)

s,t=test_data[0]
print("source:",s)
print("target:",t)
print("pred:",generate(model,s,ctx))

source: Whiskers just meowed and ran away. Nora climbed into the kayak and dipped her paddle in the water. She went around in circles, laughing and singing a silly song. Suddenly, she saw
target: Whiskers on the shore. The kitten looked sad, so Nora called out, \"Wait, Whiskers! I'll bring the kayak to you.\" Nora paddled hard, pushing the water away. She brought the kayak close
pred: the fourth branch! \"Can I have Lua the code was so happy! She asked the twinkling stars. like the time, she had to be obedient but she didn't know how. she had


## Later switches

In [ ]:
# Examples for later experiments
# cfg["use_wq"]=True;cfg["use_aq"]=True
# cfg["use_ws"]=True;cfg["ws_ratio"]=0.5
# cfg["attn_top_k"]=8